In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Window-Function-Demo'). \
    master('yarn'). \
    getOrCreate()

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Date-Time-Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [5]:
sc = spark.sparkContext

By default, PySpark assumes that each line in your file is a **complete, independent JSON object**. This is known as JSON Lines (JSONL). If your JSON spans multiple lines (with pretty-printing, nested arrays, or indentation), Spark will fail to read it correctly and likely return a single column of `null` values or a corrupt record error.

To handle this, you must explicitly enable the `multiLine` option.

# 1. The Scenario

Imagine you have a file named `movie_data.json` that looks like this:

In [6]:
! cat /home/talentum/test-jupyter/DAWPPS/json-usecases/movie_data.json

[
  {
    "id": 1,
    "movie": {
      "Title": "Inception",
      "Details": {
        "Director": "Christopher Nolan",
        "Year": 2010
      }
    }
  },
  {
    "id": 2,
    "movie": {
      "Title": "The Matrix",
      "Details": {
        "Director": "The Wachowskis",
        "Year": 1999
      }
    }
  }
]

# 2. Working Example: Loading the Data

When loading this, the `.option("multiLine", "true")` is the "magic switch" that tells Spark to parse the entire file (or the contents between the array brackets `[]`) as a single entity.

In [9]:
import pyspark.sql.functions as F

# Load the multiline JSON
df = spark.read \
    .option("multiLine", "true") \
    .json("file:///home/talentum/test-jupyter/DAWPPS/json-usecases/movie_data.json")

# Show the schema to see the nested structure
df.printSchema()

# Extract nested data
df_final = df.select(
    F.col("id"),
    F.col("movie.Title").alias("title"),
    F.col("movie.Details.Director").alias("director")
)

df_final.show()

root
 |-- id: long (nullable = true)
 |-- movie: struct (nullable = true)
 |    |-- Details: struct (nullable = true)
 |    |    |-- Director: string (nullable = true)
 |    |    |-- Year: long (nullable = true)
 |    |-- Title: string (nullable = true)

+---+----------+-----------------+
| id|     title|         director|
+---+----------+-----------------+
|  1| Inception|Christopher Nolan|
|  2|The Matrix|   The Wachowskis|
+---+----------+-----------------+



# 3. Important Performance Warning

While multiLine solves your parsing problem, it comes with a significant trade-off in performance:

- **Standard JSON:** Spark can split a massive 10GB file into small chunks and process them in parallel across 100 different worker nodes.

- **Multiline JSON:** Because a single JSON object can span the entire file, **Spark cannot split the file**. One worker node has to read the entire file into memory to parse the structure.

- **Expert Advice:** If you have thousands of small multiline JSON files, Spark will handle them fine by distributing the files. However, if you have one **giant** (e.g., 5GB+) multiline JSON file, you may run into `OutOfMemory` errors. In that case, it is better to pre-process the data into a single-line format before bringing it into Spark.

# 4. Summary of Options

<img src="images/summary_comparison3.jpg" align = "left">

In [ ]:
# Since you mentioned earlier that your data had single quotes, you can actually combine these:

df = spark.read \
    .option("multiLine", "true") \
    .option("allowSingleQuotes", "true") \
    .json("path_to_file.json")

Does your multiline data come as one large array [...] in a single file, or are you reading a directory full of separate multiline files? Then prompt the AI and see the setails